In [104]:
import numpy as np

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp

from qiskit_aer.primitives import EstimatorV2 as Estimator

from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [105]:
from torch.utils.data import DataLoader, TensorDataset

def build_qnn_model(n_features=9, reps=1, entanglement="ring"):
    n_qubits = n_features

    x_params = ParameterVector("x", n_features)
    theta_params = ParameterVector("θ", length=2 * n_qubits * reps)

    qc = QuantumCircuit(n_qubits)

    # Feature encoding: (Ry, Rz) per feature
    for i in range(n_qubits):
        qc.ry(x_params[i], i)
        qc.rz(x_params[i], i)

    t = 0
    for _ in range(reps):
        # Trainable rotations
        for q in range(n_qubits):
            qc.ry(theta_params[t], q); t += 1
            qc.rz(theta_params[t], q); t += 1

        # Entanglement
        if entanglement == "ring":
            for q in range(n_qubits - 1):
                qc.cx(q, q + 1)
            qc.cx(n_qubits - 1, 0)
        elif entanglement == "all":
            # all-to-all, one CX per unordered pair
            for i in range(n_qubits):
                for j in range(i + 1, n_qubits):
                    qc.cx(i, j)
        else:
            raise ValueError("entanglement must be 'ring' or 'all'")

    # Observable: Z on qubit 0 (matches your working snippet)
    observable = SparsePauliOp.from_list([("Z" + "I"*(n_qubits-1), 1.0)])

    estimator = Estimator()

    # IMPORTANT: pass a single observable, not a list, for your environment
    qnn = EstimatorQNN(
        circuit=qc,
        estimator=estimator,
        observables=observable,
        input_params=list(x_params),
        weight_params=list(theta_params),
    )

    # TorchConnector makes the QNN trainable in PyTorch
    qnn_torch = TorchConnector(qnn)

    # Add a small classical head to map [-1,1] -> raw fire counts (>=0)
    # softplus ensures non-negative output without scaling y.
    model = nn.Sequential(
        qnn_torch,          # (batch, 1) in [-1,1]
        nn.Linear(1, 1),    # learn affine mapping
        nn.Softplus()       # enforce y_pred >= 0
    )

    return model, qc


def train_minibatch(model, X_train_a, y_train_raw, batch_size=256, epochs=5, lr=0.01):
    torch.manual_seed(0)

    # Keep on CPU: Qiskit evaluation is CPU-bound and TorchConnector may not accept CUDA tensors.
    Xtr = torch.tensor(X_train_a, dtype=torch.float32)
    ytr = torch.tensor(y_train_raw, dtype=torch.float32).view(-1, 1)

    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)

    # For counts, Poisson loss is often better than MSE; start with Poisson.
    loss_fn = nn.PoissonNLLLoss(log_input=False, full=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()
    for epoch in range(epochs):
        total = 0.0
        n = 0
        for xb, yb in loader:
            optimizer.zero_grad()
            yhat = model(xb)
            loss = loss_fn(yhat, yb)
            loss.backward()
            optimizer.step()
            total += loss.item() * xb.shape[0]
            n += xb.shape[0]
        print(f"epoch={epoch+1:3d}, loss={total/n:.6f}")

    return model


def predict(model, X_test_a):
    model.eval()
    Xte = torch.tensor(X_test_a, dtype=torch.float32)
    with torch.no_grad():
        y_pred = model(Xte).numpy().ravel()
    return y_pred

In [106]:
class FireCountModel(nn.Module):
    """
    QNN outputs q(x) in [-1,1].
    Then a small classical head maps it to raw fire count >= 0.
    """
    def __init__(self, qnn):
        super().__init__()
        self.qnn_layer = TorchConnector(qnn)      # outputs shape (batch, 1)

        # Linear readout: y_raw = softplus(a*q + b)
        self.readout = nn.Linear(1, 1)
        self.softplus = nn.Softplus()

    def forward(self, X):
        q = self.qnn_layer(X)          # (batch, 1), roughly in [-1,1]
        y = self.softplus(self.readout(q))  # (batch, 1), >= 0
        return y

In [107]:
def train_model(model, X_train_a, y_train_raw, batch_size=256, epochs=10, lr=0.01):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("torch device:", device)

    # IMPORTANT: Qiskit evaluation will still be CPU-bound; CUDA helps only for readout math.
    model = model.to(device)

    Xtr = torch.tensor(X_train_a, dtype=torch.float32)      # keep on CPU to avoid connector CUDA issues
    ytr = torch.tensor(y_train_raw, dtype=torch.float32).view(-1, 1)

    loader = DataLoader(
        TensorDataset(Xtr, ytr),
        batch_size=batch_size,
        shuffle=True,
        drop_last=False
    )

    loss_fn = nn.PoissonNLLLoss(log_input=False, full=True)  # good for count targets
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()
    for epoch in range(epochs):
        running = 0.0
        n = 0
        for xb, yb in loader:
            # Move batch to device for the classical head; xb may need to remain CPU for QNN
            # We'll try to keep both on device, but if you see CUDA->numpy errors, keep xb on CPU.
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            yhat = model(xb)
            loss = loss_fn(yhat, yb)
            loss.backward()
            optimizer.step()

            running += loss.item() * xb.shape[0]
            n += xb.shape[0]
        print("y_train stats:", float(np.min(y_train_raw)), float(np.max(y_train_raw)), float(np.mean(y_train_raw)))
        print("ytr tensor stats:", float(ytr.min()), float(ytr.max()), float(ytr.mean()))
        print(f"epoch {epoch+1:02d}  loss {running/n:.6f}")

    return model

In [108]:
def predict(model, X_test_a):
    model.eval()
    device = next(model.parameters()).device

    Xte = torch.tensor(X_test_a, dtype=torch.float32).to(device)
    with torch.no_grad():
        y_pred = model(Xte).detach().cpu().numpy().ravel()
    return y_pred

In [ ]:
import pandas as pd

n_samples = 2000  # subset — full 155K rows is impractical for QNN

# Shuffle before sampling so we get a mix of fire/no-fire rows
df_full = pd.read_csv("features.csv").sample(n=n_samples, random_state=42).reset_index(drop=True)
df_x    = df_full[["month_sin","month_cos","year","avg_tmax_c","temp_range",
                    "tot_prcp_mm","dryness_3m_avg","latitude","longitude"]]

x = df_x.values.astype(float)
y = df_full["target"].values.astype(float)

# split and scale
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.1, random_state=0)

x_scaler = StandardScaler()
x_train_s = x_scaler.fit_transform(x_train)
x_test_s  = x_scaler.transform(x_test)

angle_scaler = MinMaxScaler(feature_range=(0, 2 * np.pi))
x_train_a = angle_scaler.fit_transform(x_train_s)
x_test_a  = angle_scaler.transform(x_test_s)

In [110]:
# n_features = 9 assumed by you; reps=1 is a good start
model, qc = build_qnn_model(n_features=9, reps=2, entanglement="ring")  # or "all"

# Train on raw y (no scaling)
model = train_minibatch(model, x_train_a, y_train_raw=y_train, batch_size=256, epochs=5, lr=0.01)

# Predict raw expected fires
y_pred = predict(model, x_test_a)

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


epoch=  1, loss=0.465204
epoch=  2, loss=0.437090
epoch=  3, loss=0.410607
epoch=  4, loss=0.385812
epoch=  5, loss=0.362858


In [135]:
import numpy as np
import torch
from sklearn.metrics import mean_squared_error, r2_score, roc_auc_score


def evaluate_regression_and_auc(
    model,
    X_test_a,
    y_test_raw,
    fire_threshold,
    batch_size=512,
):
    """
    Computes:
      - MSE and R^2 on raw regression target y_test_raw
      - AUC-ROC on a binarized label: y_test_raw > fire_threshold

    Parameters
    ----------
    model : torch.nn.Module
        Your trained model (e.g., nn.Sequential(TorchConnector(qnn), ...)).
    X_test_a : np.ndarray
        Test features (the same representation you trained on), shape (n, n_features).
    y_test_raw : np.ndarray
        Raw regression targets (expected number of fires), shape (n,).
    fire_threshold : float
        Threshold to define positive class for AUC (e.g., 0.0 or 0.5 or 1.0).
    batch_size : int
        Batch size used for prediction to reduce memory usage.

    Returns
    -------
    metrics : dict
        {'mse': ..., 'r2': ..., 'auc_roc': ...}
    y_pred : np.ndarray
        Raw predictions, shape (n,).
    """

    model.eval()

    Xte = torch.tensor(X_test_a, dtype=torch.float32)
    n = Xte.shape[0]

    preds = []
    with torch.no_grad():
        for start in range(0, n, batch_size):
            xb = Xte[start:start + batch_size]
            yb_hat = model(xb)  # shape (batch, 1)
            preds.append(yb_hat.detach().cpu().numpy())

    y_pred = np.vstack(preds).ravel()

    # --- Regression metrics (raw units) ---
    mse = mean_squared_error(y_test_raw, y_pred)
    r2 = r2_score(y_test_raw, y_pred)

    # --- AUC-ROC ---
    # Binarize the true target (do NOT filter rows; keep all samples)
    y_true_bin = (np.asarray(y_test_raw) > fire_threshold).astype(int)

    # For AUC, you need a continuous "score". y_pred works.
    # roc_auc_score requires both classes present in y_true_bin:
    if len(np.unique(y_true_bin)) < 2:
        auc = np.nan  # undefined if only one class in test set
    else:
        auc = roc_auc_score(y_true_bin, y_pred)

    return {"mse": mse, "r2": r2, "auc_roc": auc}, y_pred

In [136]:
for i in range(0,70):
    metrics, y_pred = evaluate_regression_and_auc(
    model=model,
    X_test_a=x_test_a,
    y_test_raw=y_test,
    fire_threshold=i/100,
    batch_size=512
    )
    print(metrics)

{'mse': 0.11552482727977136, 'r2': -47.33161648967771, 'auc_roc': 0.4824120603015075}
{'mse': 0.11545591332910334, 'r2': -47.30278526168194, 'auc_roc': 0.4120603015075377}
{'mse': 0.11543569459401538, 'r2': -47.29432643795385, 'auc_roc': 0.42713567839195976}
{'mse': 0.11558464823231139, 'r2': -47.3566435198347, 'auc_roc': 0.38190954773869346}
{'mse': 0.11557108672065539, 'r2': -47.35096984954398, 'auc_roc': 0.4623115577889447}
{'mse': 0.11549995941277647, 'r2': -47.32121262897682, 'auc_roc': 0.5025125628140703}
{'mse': 0.11546959187383848, 'r2': -47.30850789458988, 'auc_roc': 0.33668341708542715}
{'mse': 0.11554595915152116, 'r2': -47.340457338395424, 'auc_roc': 0.42211055276381915}
{'mse': 0.11549612748929727, 'r2': -47.319609483918086, 'auc_roc': 0.38190954773869346}
{'mse': 0.11541916128590816, 'r2': -47.28740947018545, 'auc_roc': 0.33668341708542715}
{'mse': 0.1154454623746286, 'r2': -47.298412941588346, 'auc_roc': 0.40703517587939697}
{'mse': 0.11534284378138374, 'r2': -47.2554808

In [137]:
y_mean = float(np.mean(y_train))
y_nonzero_rate = float(np.mean(y_train > 0))
print("mean(y):", y_mean)
print("P(y>0):", y_nonzero_rate)

# If model predicts constant c ~0.88, compare its MSE to baseline mean predictor:
c = 0.88
mse_const = np.mean((y_train - c)**2)
mse_mean = np.mean((y_train - y_mean)**2)
print("MSE const(0.88):", mse_const)
print("MSE mean:", mse_mean)


mean(y): 0.008922315660297291
P(y>0): 0.011111111111111112
MSE const(0.88): 0.7664438190243558
MSE mean: 0.007667486869736899


In [139]:
import numpy as np
import torch

def print_all_trained_params(model, max_values_per_param=None):
    """
    Prints a full list of *all* trainable parameters in `model`.

    max_values_per_param:
      - None  -> print ALL values (can be huge for QNN weights)
      - int   -> print only first N values per parameter tensor
    """
    named = list(model.named_parameters())
    if not named:
        print("No parameters found (model.named_parameters() is empty).")
        return

    total = 0
    print("=== Trainable parameters (full list) ===")
    for i, (name, p) in enumerate(named):
        if not p.requires_grad:
            continue

        v = p.detach().cpu().numpy()
        total += p.numel()

        print(f"\n[{i}] name: {name}")
        print(f"    shape: {tuple(p.shape)}  dtype: {p.dtype}  numel: {p.numel()}")

        flat = v.reshape(-1)
        if max_values_per_param is None:
            # print all values
            np.set_printoptions(threshold=np.inf, linewidth=200)
            print("    values:", flat)
        else:
            print(f"    values (first {max_values_per_param}):", flat[:max_values_per_param])

    print(f"\nTotal trainable scalars: {total}")
# Print absolutely everything (warning: can be a lot)
print_all_trained_params(model, max_values_per_param=None)

# Safer: print only first 50 values per parameter tensor
# print_all_trained_params(model, max_values_per_param=50)

=== Trainable parameters (full list) ===

[0] name: 0.weight
    shape: (36,)  dtype: torch.float32  numel: 36
    values: [-0.50961024  0.5785481   0.00532273  0.9159788  -0.0804912   0.43700486 -0.01502834 -0.22887488 -0.57495624 -0.4739551  -0.72146535  0.2603914   0.61370945  0.5746634  -0.7407316  -0.69496167
  0.55740124  1.1246884  -0.17415693  0.74831176 -0.3863102   0.1058141   0.61649066 -0.92767036 -0.68760896 -0.25316525 -0.5054633   0.8640008  -0.89296484 -0.46033287 -0.66502476 -0.936561
 -0.87942666  0.85959804  0.44147253  0.48467255]

[1] name: 1.weight
    shape: (1, 1)  dtype: torch.float32  numel: 1
    values: [0.33071044]

[2] name: 1.bias
    shape: (1,)  dtype: torch.float32  numel: 1
    values: [-0.90045804]

Total trainable scalars: 38


In [143]:
import numpy as np
import torch

def predict_expected_fires(model, X_all_a, batch_size=1024):
    """
    Returns expected-fire-count predictions for every row in X_all_a.

    model should output raw fire counts (e.g., includes Linear + Softplus head).
    """
    model.eval()

    X = torch.tensor(X_all_a, dtype=torch.float32)
    n = X.shape[0]

    preds = []
    with torch.no_grad():
        for start in range(0, n, batch_size):
            xb = X[start:start + batch_size]
            yb = model(xb)  # (batch, 1)
            preds.append(yb.detach().cpu().numpy())

    y_pred_all = np.vstack(preds).ravel()
    return y_pred_all

df = pd.read_csv("features.csv")
dfx = df[["month_sin","month_cos","year","avg_tmax_c","temp_range",
                    "tot_prcp_mm","dryness_3m_avg","latitude","longitude"]]
x_a = df_x.values.astype(float)
#y_a = df_full["target"].values.astype(float)

y_expected_all = predict_expected_fires(model, x_a, batch_size=1024)


print(y_expected_all.shape)
print("first 10 predictions:", y_expected_all[:1000])
print(min(y_expected_all))

(2000,)
first 10 predictions: [0.34521213 0.34480035 0.34273943 0.34737957 0.34731737 0.34449458 0.34274462 0.34824595 0.3434786  0.34815732 0.34877086 0.34884953 0.34089315 0.3466594  0.3477157  0.34642825 0.3470671  0.35175017
 0.3493054  0.34496796 0.34149644 0.34949303 0.34736603 0.34725842 0.34340802 0.3448358  0.34547952 0.34489852 0.34083542 0.347197   0.34620878 0.3456153  0.341389   0.34385446 0.33949697 0.34890848
 0.34454277 0.34193006 0.3486188  0.34448668 0.34252912 0.34997305 0.34554094 0.3419822  0.3480332  0.3444136  0.3437829  0.3425718  0.34627667 0.34438315 0.3466006  0.3461455  0.34218812 0.34961486
 0.34307933 0.3435822  0.3473538  0.3424603  0.34150222 0.34482726 0.33942553 0.34370336 0.34169376 0.34452465 0.34405568 0.34426698 0.348966   0.33984447 0.34285212 0.34386885 0.34549406 0.34626323
 0.34437245 0.34502298 0.33869225 0.35047108 0.33992618 0.34335467 0.34186482 0.34303352 0.34621122 0.34550723 0.3466522  0.3495956  0.34894615 0.34383884 0.34549952 0.347080